In [3]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[-0.2845],
        [-0.3718]], grad_fn=<AddmmBackward0>)

In [4]:
print(net[2].state_dict())

OrderedDict({'weight': tensor([[ 0.2244,  0.2998, -0.0712, -0.2983,  0.1709, -0.0605,  0.1847,  0.0612]]), 'bias': tensor([-0.2358])})


In [5]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.2358], requires_grad=True)
tensor([-0.2358])


由于我们还没有调用反向传播，所以参数的梯度处于初始状态。

In [6]:
net[2].weight.grad == None

True

PyTorch 的 nn.Linear 从张量形状来看，使用的是：
\[
Y=XW^T+b
\]def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

In [7]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [8]:
net.state_dict()['2.bias'].data

tensor([-0.2358])

In [ ]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net

# block2相当于整个第0层，而nn.Linear(4, 1)相当于第1层，所以block2()的输出是4维，nn.Linear(4, 1)的输入是4维，输出是1维
# 在这个第0层内部，block1()重复四次
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.0806],
        [-0.0806]], grad_fn=<AddmmBackward0>)

In [10]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [11]:
rgnet[0][1][0].bias.data

tensor([-0.0368, -0.3601,  0.2383,  0.3066,  0.0412, -0.4832,  0.2124,  0.0748])

In [14]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
print(net[0].weight)
print(net[0].bias)
net[0].weight.data[0], net[0].bias.data[0]

Parameter containing:
tensor([[ 0.0135,  0.0109, -0.0088,  0.0057],
        [ 0.0110,  0.0017,  0.0054,  0.0035],
        [ 0.0014,  0.0136,  0.0009, -0.0012],
        [-0.0079,  0.0015, -0.0013,  0.0032],
        [-0.0081, -0.0056, -0.0256,  0.0020],
        [-0.0009,  0.0038, -0.0124,  0.0152],
        [-0.0031, -0.0035, -0.0035,  0.0097],
        [-0.0105, -0.0053,  0.0021,  0.0229]], requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)


(tensor([ 0.0135,  0.0109, -0.0088,  0.0057]), tensor(0.))

In [15]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [19]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net)
print(net[0].weight.data)
print(net[0].bias.data)
print(net[0].weight.data[0])
print(net[2].weight.data)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)
tensor([[-0.4493,  0.2947,  0.3358, -0.3366],
        [-0.3638,  0.1823, -0.6713,  0.4697],
        [ 0.3416,  0.1276,  0.4140, -0.6724],
        [ 0.6911,  0.2705, -0.4048, -0.3841],
        [ 0.6894, -0.0590, -0.3804, -0.1430],
        [ 0.6547, -0.4992,  0.0198,  0.1149],
        [ 0.4623, -0.7027,  0.6412,  0.4387],
        [ 0.5741, -0.2631, -0.1274,  0.2762]])
tensor([0., 0., 0., 0., 0., 0., 0., 0.])
tensor([-0.4493,  0.2947,  0.3358, -0.3366])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [20]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 6.9194, -0.0000,  0.0000,  9.6442],
        [ 7.0029, -0.0000,  0.0000, -7.2892]], grad_fn=<SliceBackward0>)

In [25]:
print(net[0].weight.data[:])#和net[0].weight.data是同一个对象
net[0].weight.data[:] += 1 
print(net[0].weight.data[:])
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([[42.0000,  3.0000,  3.0000, 12.6442],
        [10.0029,  3.0000,  3.0000, -4.2892],
        [12.1047,  3.0000,  3.0000,  3.0000],
        [ 3.0000,  3.0000, -6.1130,  9.1662],
        [12.7542, 12.1435,  3.0000,  3.0000],
        [ 3.0000, 11.1399, 12.8657,  3.0000],
        [ 3.0000,  3.0000,  3.0000, 10.1722],
        [12.7535, -6.6963, 12.3007, -3.9118]])
tensor([[43.0000,  4.0000,  4.0000, 13.6442],
        [11.0029,  4.0000,  4.0000, -3.2892],
        [13.1047,  4.0000,  4.0000,  4.0000],
        [ 4.0000,  4.0000, -5.1130, 10.1662],
        [13.7542, 13.1435,  4.0000,  4.0000],
        [ 4.0000, 12.1399, 13.8657,  4.0000],
        [ 4.0000,  4.0000,  4.0000, 11.1722],
        [13.7535, -5.6963, 13.3007, -2.9118]])


tensor([42.0000,  4.0000,  4.0000, 13.6442])

In [26]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
